In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))

In [ ]:
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
text1 = "Erreur lors du chargement du tableau de bord"
text2 = "Le tableau de bord ne s'affiche pas"
text3 = "Le client a commandé un nouveau serveur"

In [ ]:
v1 = emb.embed_query(text1)
v2 = emb.embed_query(text2)
v3 = emb.embed_query(text3)

In [ ]:
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos
cos ([v1], [v2]) [0][0]

In [ ]:
print("Sim(text1, text2) =", cos([v1], [v2])[0][0])
print("Sim(text1, text3) =", cos([v1], [v3])[0][0])
print("Sim(text2, text3) =", cos([v2], [v3])[0][0])

In [ ]:
sentences = [
    "Erreur lors du chargement du tableau de bord.",
    "Le tableau de bord ne s'affiche pas.",
    "Impossible de se connecter à l'API.",
    "Problème d'authentification sur le service.",
    "Le client a commandé un nouveau serveur.",
    "Livraison prévue pour la semaine prochaine.",
    "Le serveur physique est tombé en panne.",
    "Mise à jour réussie du système."
]

In [ ]:
vectors = [emb.embed_query(sentence) for sentence in sentences]

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
   
vectors_np = np.array(vectors)
tsne = TSNE(n_components=2, perplexity=5, random_state=2)
points_tsne = tsne.fit_transform(vectors_np)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(points_tsne[:, 0], points_tsne[:, 1])

for i, txt in enumerate(sentences):
    plt.annotate(txt, (points_tsne[i, 0], points_tsne[i, 1]))
    
plt.title("t-SNE Visualization of Sentence Embeddings")
plt.show()

In [ ]:
import umap
   
reducer = umap.UMAP(n_components=2, random_state=42)
points_umap = reducer.fit_transform(vectors)
   
plt.figure(figsize=(8,6))
plt.scatter(points_umap[:,0], points_umap[:,1])
   
for i, txt in enumerate(sentences):
        plt.annotate(txt, (points_umap[i,0], points_umap[i,1]))
   
plt.title("UMAP - Visualisation des embeddings (2D)")
plt.show()

In [ ]:
# Charger un modèle local - all-MiniLM-L6-v2
   
from sentence_transformers import SentenceTransformer
model_local = SentenceTransformer("all-MiniLM-L6-v2")

text1_local = "le tableau de bord ne se charge plus."
vec_local = model_local.encode(text1_local)
vec_local[:10]


In [ ]:
sentences = [
        "Erreur lors du chargement du tableau de bord.",
       "Le tableau de bord ne s'affiche pas.",
       "Le client a commandé un nouveau serveur."
    ]
vecs = model_local.encode(sentences)
vecs

Document loading

In [ ]:
import os
   
docs_directory = "../data/text_files"
raw_docs = []
filenames = []
metadatas = []
   
for filename in os.listdir(docs_directory):
        if filename.endswith(".txt"):
            path = os.path.join(docs_directory, filename)
            with open(path, "r", encoding='utf-8') as f :
                text = f.read()
                raw_docs.append(text)
                filenames.append(filename)
                metadatas.append({"source":filename})

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
   
splitter = RecursiveCharacterTextSplitter(
        chunk_size = 600,
        chunk_overlap = 80,
        separators=["\n\n", "\n", ".", " ", ""]
    )

all_chunks = splitter.create_documents(raw_docs, metadatas=metadatas)
print(len(all_chunks))
print(all_chunks[0].page_content)

Embedding(génération des vecteurs)

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Charger ta clé API depuis le .env
import os
from dotenv import load_dotenv
 
load_dotenv()
   
os.getenv("OPENAI_API_KEY")[:10] + "..."

In [ ]:
# Initialiser le modèle d’embeddings
emb = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# Générer tes premiers embeddings
sample_vector = emb.embed_query(all_chunks[0].page_content)
print(len(sample_vector))

In [ ]:
from langchain_chroma import Chroma
 
from langchain_core.documents import Document

In [ ]:
documents = []

for chunk, meta in zip(all_chunks, metadatas):
    documents.append(Document(page_content=chunk.page_content, metadata={"source": meta["source"]}))

In [ ]:
# Initialiser ChromaDB comme Vector Store
   
from langchain_community.vectorstores import Chroma
vector_store = Chroma(
        collection_name= "support_docs",
        embedding_function=emb,
        persist_directory="./vector_store"
    )

In [ ]:
vector_store.add_documents(documents)

In [ ]:
query = "Quelle est la cause des retards observés dans le tableau de bord Analytics ?"

In [ ]:
results = vector_store.similarity_search("pourquoi le tableau de bord ne se charge plus.", k=3)
    

In [ ]:
# Affichage des résultats
for i, doc in enumerate(results):
        print(f"\n--- RESULT {i+1} ---")
        print("Source :", doc.metadata.get("source")),
        print(doc.page_content[:300], "...")
        
 # Construire un contexte à partir des résultats Chroma
def build_context(docs) : 
        context = ""
        for doc in docs : 
            context  += f"Source :{doc.metadata.get('source')}\n"
            context += doc.page_content + "\n\n"
        return context
        
        
context = build_context(results)
print(context[:700]+"...")

In [ ]:
# Construire la question finale (Prompt Final)

final_prompt =  f"""
Tu es un assistant technique interne.
    Utilise UNIQUEMENT les informations du contexte ci-dessous pour répondre.
    Si l'information n'est pas présente , dis que tu ne sais pas.  
### CONTEXTE
    {context}

### QUESTION
    {query}

### REPONSE: 
    """
print(final_prompt[:700]+"...")

In [ ]:
# Envoyer au modèle (OPENAI / GPT)
from openai import OpenAI
client = OpenAI()
    
response = client.chat.completions.create(
        model="gpt-4o-mini",
       messages=[
            {"role": "user", "content": final_prompt}
        ]
    )
print(response.choices[0].message.content)

In [ ]:
# Question utilisateur
query = "Quelle est la cause des retards observés dans le tableau de bord Analytics ?"

In [ ]:
 # Recherche avec scores
results_with_scores = vector_store.similarity_search_with_score(query, k=5)

In [ ]:
# Afficher les résultats
for doc, score in results_with_scores:
        print("\nScore :", round(score, 3))
        print("Source :", doc.metadata.get("source"))
        print(doc.page_content[:200], "...") 

In [ ]:
THRESHOLD = 1.05  # distance max acceptable
filtered_docs = [
        doc for doc, score in results_with_scores
       if score <= THRESHOLD
]


if not filtered_docs:
        print("Aucun document pertinent trouvé : le RAG doit répondre 'Je ne sais pas'.")
else:
        print(f"{len(filtered_docs)} document(s) pertinent(s) trouvé(s).")

In [ ]:
# Construire le contexte à partir des documents pertinents 
def build_context_from_docs(docs):
        context = ""
        for i, doc in enumerate(docs, start=1):
           source = doc.metadata.get("source", "inconnu")
           context += f"--- Document {i} (source : {source}) ---\n"
           context += doc.page_content.strip() + "\n\n"
        return context

In [ ]:
# Construire le prompt final (LLM-ready)
def build_final_prompt(query: str, context: str) -> str:
        return f"""
    Tu es un assistant interne pour une équipe Data & Analytics.

    Tu dois répondre UNIQUEMENT à partir du contexte ci-dessous.
    Si la réponse ne se trouve pas dans le contexte, dis explicitement : 
    "Je ne sais pas, l'information n'est pas présente dans les documents."
### CONTEXTE
    {context}
    
    ### QUESTION UTILISATEUR
    {query}
    
    ### RÉPONSE ATTENDUE
    - Sois clair et concis.
    - Base-toi uniquement sur le contexte.
    - Si possible, résume en 3 à 5 phrases maximum.
    """

In [ ]:
from openai import OpenAI
client = OpenAI()
if not filtered_docs:
        print("Aucun document suffisamment pertinent trouvé.")
        print("Réponse RAG : Je ne sais pas, l'information " \
        "n'est pas présente dans les documents.")
else:
       context = build_context_from_docs(filtered_docs)
       final_prompt = build_final_prompt(query, context)

       response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
               {"role": "user", "content": final_prompt}
           ]
       )
answer = response.choices[0].message.content
print("🔎 QUESTION :", query)
print("\n📄 CONTEXTE UTILISÉ :\n")
print(context[:800] + "...\n")
print("✅ RÉPONSE DU LLM :\n")
print(answer)